# 🍇 GRAPE — INTERNAL + EXTERNAL DATA PREPROCESSING
### Project: Plant Disease Detection (Computer Vision)
**Output:** `3rd Preprocessing/grape_processed_data.npz`  
**Resolution:** `224 x 224 x 3` (RGB uint8)  
**Classes (3):** `Healthy (0)`, `Black_Rot (1)`, `Leaf_Blight (2)`  
**Standard:** MobileNetV2 uint8 [0, 255] internal normalization pipeline (consistent with Tomato, Apple & Corn).



In [3]:
# ================================================================
# 🍇 GRAPE — STEP 1: LOAD RAW VERIFIED IMAGES & BUILD NPZ ARRAY
# ================================================================

import os
import random
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT = '/content/drive/MyDrive/Plant Disease Detection (Computer Vision)'
if not os.path.exists(PROJECT):
    PROJECT = r'G:\My Drive\Plant Disease Detection (Computer Vision)'

RAW_DIR = os.path.join(PROJECT, '1st Raw_Data', 'Grape')
INTERNAL_DIR = os.path.join(RAW_DIR, 'Internal_PlantVillage')
EXTERNAL_DIR = os.path.join(RAW_DIR, 'External_Natural')
PREPROCESS_DIR = os.path.join(PROJECT, '3rd Preprocessing')
os.makedirs(PREPROCESS_DIR, exist_ok=True)

NPZ_OUT = os.path.join(PREPROCESS_DIR, 'grape_processed_data.npz')

CLASSES = ['Healthy', 'Black_Rot', 'Leaf_Blight']
IMG_SIZE = (224, 224)
SEED = 42

print('=' * 75)
print('🍇 GRAPE PREPROCESSING PIPELINE')
print('=' * 75)
print('Classes:', CLASSES)
print('Target Size:', IMG_SIZE)

# Collect all image paths and labels
image_records = []

for src_name, src_dir in [('Internal_PlantVillage', INTERNAL_DIR), ('External_Natural', EXTERNAL_DIR)]:
    for class_idx, class_name in enumerate(CLASSES):
        folder = os.path.join(src_dir, class_name)
        if not os.path.exists(folder):
            continue
        files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
        for f in files:
            image_records.append({
                'path': os.path.join(folder, f),
                'class_name': class_name,
                'class_idx': class_idx,
                'source': src_name
            })

df = pd.DataFrame(image_records)
print(f'\nTotal raw images discovered: {len(df)}')
if len(df) == 0:
    print('⚠️ No images found in Grape Raw_Data yet. Please run Stage 2 Collection first.')
else:
    print(df.groupby(['source', 'class_name']).size())

    # Process into uint8 arrays
    X_list = []
    y_list = []
    source_list = []

    print('\nResizing and converting images to 224x224 RGB uint8...')
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        try:
            with Image.open(row['path']) as img:
                img = img.convert('RGB')
                img = img.resize(IMG_SIZE, Image.Resampling.BILINEAR)
                arr = np.array(img, dtype=np.uint8)
                X_list.append(arr)
                y_list.append(row['class_idx'])
                source_list.append(row['source'])
        except Exception as e:
            print(f"Skipping corrupted: {row['path']} -> {e}")

    X = np.array(X_list, dtype=np.uint8)
    y = np.array(y_list, dtype=np.int64)
    source = np.array(source_list)
    class_names = np.array(CLASSES)

    print(f'\nArray Shapes:')
    print(f'  X: {X.shape}, dtype={X.dtype}')
    print(f'  y: {y.shape}, dtype={y.dtype}')
    print(f'  source: {source.shape}')

    # Save to compressed NPZ
    print(f'\nSaving to {NPZ_OUT}...')
    np.savez_compressed(
        NPZ_OUT,
        X=X,
        y=y,
        class_names=class_names,
        source=source
    )
    print(f'✅ Successfully saved grape_processed_data.npz! Size: {round(os.path.getsize(NPZ_OUT)/(1024**2), 2)} MB')



Mounted at /content/drive
🍇 GRAPE PREPROCESSING PIPELINE
Classes: ['Healthy', 'Black_Rot', 'Leaf_Blight']
Target Size: (224, 224)

Total raw images discovered: 1800
source                 class_name 
External_Natural       Black_Rot      300
                       Healthy        300
                       Leaf_Blight    300
Internal_PlantVillage  Black_Rot      300
                       Healthy        300
                       Leaf_Blight    300
dtype: int64

Resizing and converting images to 224x224 RGB uint8...


  0%|          | 0/1800 [00:00<?, ?it/s]


Array Shapes:
  X: (1800, 224, 224, 3), dtype=uint8
  y: (1800,), dtype=int64
  source: (1800,)

Saving to /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/grape_processed_data.npz...
✅ Successfully saved grape_processed_data.npz! Size: 183.24 MB


In [4]:
# ================================================================
# 🍇 GRAPE — STEP 2: LOAD AND AUDIT PROCESSED NPZ DATASET
# ================================================================

import os
import numpy as np

BASE = '/content/drive/MyDrive/Plant Disease Detection (Computer Vision)'
if not os.path.exists(BASE):
    BASE = r'G:\My Drive\Plant Disease Detection (Computer Vision)'

NPZ_PATH = os.path.join(BASE, '3rd Preprocessing', 'grape_processed_data.npz')

if os.path.exists(NPZ_PATH):
    data = np.load(NPZ_PATH)
    X = data['X']
    y = data['y']
    class_names = data['class_names']
    source = data['source']

    print('=' * 75)
    print('🍇 GRAPE PROCESSED DATASET AUDIT')
    print('=' * 75)
    print(f'X shape      : {X.shape} (dtype: {X.dtype})')
    print(f'y shape      : {y.shape} (dtype: {y.dtype})')
    print(f'class_names  : {class_names}')
    print(f'source shape : {source.shape}')
    print(f'Total images : {len(X)}')

    print('\nClass Counts:')
    for i, c in enumerate(class_names):
        print(f'  {c:<18}: {np.sum(y == i)}')

    print('\nSource Distribution:')
    for s in np.unique(source):
        print(f'  {s:<22}: {np.sum(source == s)}')

    print('\n✅ Grape Preprocessing Verified & Locked.')
else:
    print(f'NPZ not found at: {NPZ_PATH}. Execute Step 1 cell to generate.')



🍇 GRAPE PROCESSED DATASET AUDIT
X shape      : (1800, 224, 224, 3) (dtype: uint8)
y shape      : (1800,) (dtype: int64)
class_names  : ['Healthy' 'Black_Rot' 'Leaf_Blight']
source shape : (1800,)
Total images : 1800

Class Counts:
  Healthy           : 600
  Black_Rot         : 600
  Leaf_Blight       : 600

Source Distribution:
  External_Natural      : 900
  Internal_PlantVillage : 900

✅ Grape Preprocessing Verified & Locked.
